# Simulating a Phospholipid Bilayer

Phospholipid bilayers are fundamental structures in biological membranes and are commonly studied using neutron and X-ray reflectometry.
In this tutorial, we will explore how to use the `Bilayer` assembly in `easyreflectometry` to model a lipid bilayer structure.

A bilayer consists of two surfactant layers arranged in an inverted configuration:

```
Head₁ - Tail₁ - Tail₂ - Head₂
```

The `Bilayer` assembly comes with pre-configured constraints that represent physically meaningful relationships:
- Both tail layers share the same structural parameters
- Head layers share thickness and area per molecule (but can have different hydration)
- Conformal roughness across all interfaces

## Setup

First, we import the necessary modules and configure matplotlib for inline plotting.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import easyreflectometry
from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.sample import Bilayer
from easyreflectometry.sample import LayerAreaPerMolecule
from easyreflectometry.sample import Material
from easyreflectometry.sample import Layer
from easyreflectometry.sample import Multilayer
from easyreflectometry.sample import Sample
from easyreflectometry.model import Model
from easyreflectometry.model import PercentageFwhm

## Creating a Bilayer

We'll create a DPPC (dipalmitoylphosphatidylcholine) bilayer, a common model phospholipid.

First, let's define the materials for our solvents - D₂O (heavy water) for the head groups and air for the tail groups.

In [ ]:
# Define solvent materials
d2o = Material(sld=6.36, isld=0, name='D2O')
air = Material(sld=0, isld=0, name='Air')

### Creating Layer Components

Now we create the head and tail layers using `LayerAreaPerMolecule`. This approach allows us to define layers based on their chemical formula and area per molecule, which provides a more physically meaningful parameterization.

In [ ]:
# Create a head layer for the bilayer
# The head group of DPPC has formula C10H18NO8P
head_layer = LayerAreaPerMolecule(
    molecular_formula='C10H18NO8P',
    thickness=10.0,
    solvent=d2o,
    solvent_fraction=0.3,  # 30% solvent in head region
    area_per_molecule=48.2,
    roughness=3.0,
    name='DPPC Head'
)

# Create a tail layer for the bilayer
# The tail group of deuterated DPPC has formula C32D64
tail_layer = LayerAreaPerMolecule(
    molecular_formula='C32D64',
    thickness=16.0,
    solvent=air,
    solvent_fraction=0.0,  # No solvent in the tail region
    area_per_molecule=48.2,
    roughness=3.0,
    name='DPPC Tail'
)

### Creating the Bilayer Assembly

Now we create the `Bilayer` assembly. The bilayer will automatically:
- Create a second tail layer with parameters constrained to the first
- Create a back head layer with thickness and area per molecule constrained to the front head
- Apply conformal roughness across all layers

In [ ]:
# Create the bilayer with default constraints
bilayer = Bilayer(
    front_head_layer=head_layer,
    tail_layer=tail_layer,
    constrain_heads=True,      # Head layers share thickness and area per molecule
    conformal_roughness=True,  # All layers share the same roughness
    name='DPPC Bilayer'
)

print(bilayer)

## Exploring the Bilayer Structure

Let's examine the layer structure of our bilayer.

In [ ]:
# The bilayer has 4 layers
print(f'Number of layers: {len(bilayer.layers)}')
print()

# Access individual layers
print('Layer structure (from front to back):')
print(f'  1. Front Head: {bilayer.front_head_layer.name}')
print(f'  2. Front Tail: {bilayer.front_tail_layer.name}')
print(f'  3. Back Tail:  {bilayer.back_tail_layer.name}')
print(f'  4. Back Head:  {bilayer.back_head_layer.name}')

### Verifying Constraints

Let's verify that the constraints are working correctly.

In [ ]:
# Check tail layer constraints - both tails should have the same parameters
print('Tail layer parameters:')
print(f'  Front tail thickness: {bilayer.front_tail_layer.thickness.value:.2f} Å')
print(f'  Back tail thickness:  {bilayer.back_tail_layer.thickness.value:.2f} Å')
print(f'  Front tail APM: {bilayer.front_tail_layer.area_per_molecule:.2f} Å²')
print(f'  Back tail APM:  {bilayer.back_tail_layer.area_per_molecule:.2f} Å²')
print()

# Change front tail thickness - back should follow
bilayer.front_tail_layer.thickness.value = 18.0
print('After changing front tail thickness to 18 Å:')
print(f'  Front tail thickness: {bilayer.front_tail_layer.thickness.value:.2f} Å')
print(f'  Back tail thickness:  {bilayer.back_tail_layer.thickness.value:.2f} Å')

In [ ]:
# Check head layer constraints
print('Head layer parameters:')
print(f'  Front head thickness: {bilayer.front_head_layer.thickness.value:.2f} Å')
print(f'  Back head thickness:  {bilayer.back_head_layer.thickness.value:.2f} Å')
print()

# Hydration (solvent fraction) remains independent
print('Head layer hydration (independent):')
print(f'  Front head solvent fraction: {bilayer.front_head_layer.solvent_fraction:.2f}')
print(f'  Back head solvent fraction:  {bilayer.back_head_layer.solvent_fraction:.2f}')

# Change back head hydration - front should NOT change
bilayer.back_head_layer.solvent_fraction = 0.5
print()
print('After changing back head solvent fraction to 0.5:')
print(f'  Front head solvent fraction: {bilayer.front_head_layer.solvent_fraction:.2f}')
print(f'  Back head solvent fraction:  {bilayer.back_head_layer.solvent_fraction:.2f}')

In [ ]:
# Check conformal roughness
print('Roughness values (conformal):')
print(f'  Front head roughness: {bilayer.front_head_layer.roughness.value:.2f} Å')
print(f'  Front tail roughness: {bilayer.front_tail_layer.roughness.value:.2f} Å')
print(f'  Back tail roughness:  {bilayer.back_tail_layer.roughness.value:.2f} Å')
print(f'  Back head roughness:  {bilayer.back_head_layer.roughness.value:.2f} Å')
print()

# Change front head roughness - all should follow
bilayer.front_head_layer.roughness.value = 4.0
print('After changing front head roughness to 4.0 Å:')
print(f'  Front head roughness: {bilayer.front_head_layer.roughness.value:.2f} Å')
print(f'  Front tail roughness: {bilayer.front_tail_layer.roughness.value:.2f} Å')
print(f'  Back tail roughness:  {bilayer.back_tail_layer.roughness.value:.2f} Å')
print(f'  Back head roughness:  {bilayer.back_head_layer.roughness.value:.2f} Å')

## Building a Complete Sample

To simulate reflectometry, we need to create a complete sample with sub- and super-phases.

In [ ]:
# Reset bilayer parameters
bilayer.front_head_layer.roughness.value = 3.0
bilayer.front_tail_layer.thickness.value = 16.0
bilayer.back_head_layer.solvent_fraction = 0.3

# Create superphase (air) and subphase (silicon substrate with oxide layer)
vacuum = Material(sld=0, isld=0, name='Vacuum')
superphase = Layer(material=vacuum, thickness=0, roughness=0, name='Vacuum Superphase')

si = Material(sld=2.047, isld=0, name='Si')
sio2 = Material(sld=3.47, isld=0, name='SiO2')
si_layer = Layer(material=si, thickness=0, roughness=0, name='Si')
sio2_layer = Layer(material=sio2, thickness=15, roughness=3, name='SiO2')

# D2O subphase
d2o_layer = Layer(material=d2o, thickness=0, roughness=3, name='D2O Subphase')

# Create sample: superphase -> bilayer -> SiO2 -> Si
sample = Sample(
    Multilayer(superphase, name='Superphase'),
    bilayer,
    Multilayer(d2o_layer, name='D2O'),
    Multilayer([sio2_layer, si_layer], name='Substrate'),
    name='Bilayer on Si/SiO2'
)

print(sample)

## Simulating Reflectivity

Now we can simulate the reflectometry profile for our bilayer sample.

In [ ]:
# Create the model
model = Model(
    sample=sample,
    scale=1.0,
    background=1e-7,
    resolution_function=PercentageFwhm(5),
    name='Bilayer Model'
)

# Set up the calculator
interface = CalculatorFactory()
model.interface = interface

# Generate Q values
q = np.linspace(0.005, 0.3, 500)

# Calculate reflectometry
reflectivity = model.interface().reflectity_profile(q, model.unique_name)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogy(q, reflectivity, 'b-', linewidth=2, label='Bilayer')
plt.xlabel('Q (Å⁻¹)')
plt.ylabel('Reflectivity')
plt.title('Simulated Reflectometry of DPPC Bilayer')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Scattering Length Density Profile

Let's also visualize the SLD profile of our bilayer structure.

In [ ]:
# Get SLD profile
z, sld = model.interface().sld_profile(model.unique_name)

plt.figure(figsize=(10, 6))
plt.plot(z, sld, 'b-', linewidth=2)
plt.xlabel('Distance from interface (Å)')
plt.ylabel('SLD (10⁻⁶ Å⁻²)')
plt.title('SLD Profile of DPPC Bilayer on Si/SiO2')
plt.grid(True, alpha=0.3)
plt.show()

## Modifying Constraints

The bilayer constraints can be modified at runtime. Let's see how disabling conformal roughness affects the model.

In [ ]:
# Disable conformal roughness
bilayer.conformal_roughness = False

# Now we can set different roughness values for each layer
bilayer.front_head_layer.roughness.value = 2.0
bilayer.front_tail_layer.roughness.value = 1.5
bilayer.back_tail_layer.roughness.value = 1.5
bilayer.back_head_layer.roughness.value = 4.0

print('Individual roughness values after disabling conformal roughness:')
print(f'  Front head: {bilayer.front_head_layer.roughness.value:.2f} Å')
print(f'  Front tail: {bilayer.front_tail_layer.roughness.value:.2f} Å')
print(f'  Back tail:  {bilayer.back_tail_layer.roughness.value:.2f} Å')
print(f'  Back head:  {bilayer.back_head_layer.roughness.value:.2f} Å')

In [ ]:
# Compare reflectometry with different roughness configurations
reflectivity_variable_roughness = model.interface().reflectity_profile(q, model.unique_name)

# Reset to conformal roughness
bilayer.conformal_roughness = True
bilayer.front_head_layer.roughness.value = 3.0
reflectivity_conformal = model.interface().reflectity_profile(q, model.unique_name)

plt.figure(figsize=(10, 6))
plt.semilogy(q, reflectivity_conformal, 'b-', linewidth=2, label='Conformal roughness (3.0 Å)')
plt.semilogy(q, reflectivity_variable_roughness, 'r--', linewidth=2, label='Variable roughness')
plt.xlabel('Q (Å⁻¹)')
plt.ylabel('Reflectivity')
plt.title('Effect of Roughness Configuration on Reflectometry')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Asymmetric Hydration

One of the key features of the `Bilayer` assembly is the ability to model asymmetric hydration - where the two sides of the bilayer have different solvent exposure.

In [ ]:
# Create a bilayer with asymmetric hydration
# This is common in supported bilayers where one side is in contact with a substrate

# Low hydration on front (substrate side)
bilayer.front_head_layer.solvent_fraction = 0.1

# Higher hydration on back (solution side)
bilayer.back_head_layer.solvent_fraction = 0.4

print('Asymmetric hydration:')
print(f'  Front head solvent fraction: {bilayer.front_head_layer.solvent_fraction:.2f}')
print(f'  Back head solvent fraction:  {bilayer.back_head_layer.solvent_fraction:.2f}')

# Calculate new reflectometry
reflectivity_asymmetric = model.interface().reflectity_profile(q, model.unique_name)

# Reset to symmetric for comparison
bilayer.front_head_layer.solvent_fraction = 0.3
bilayer.back_head_layer.solvent_fraction = 0.3
reflectivity_symmetric = model.interface().reflectity_profile(q, model.unique_name)

plt.figure(figsize=(10, 6))
plt.semilogy(q, reflectivity_symmetric, 'b-', linewidth=2, label='Symmetric hydration (0.3/0.3)')
plt.semilogy(q, reflectivity_asymmetric, 'r--', linewidth=2, label='Asymmetric hydration (0.1/0.4)')
plt.xlabel('Q (Å⁻¹)')
plt.ylabel('Reflectivity')
plt.title('Effect of Asymmetric Hydration on Reflectometry')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

In this tutorial, we have explored the `Bilayer` assembly in `easyreflectometry`:

1. **Creating a bilayer**: Using `LayerAreaPerMolecule` components for head and tail layers
2. **Understanding constraints**: 
   - Tail layers share all structural parameters
   - Head layers share thickness and area per molecule (hydration is independent)
   - Conformal roughness applies to all layers by default
3. **Building a sample**: Combining the bilayer with sub- and super-phases
4. **Simulating reflectometry**: Using the calculator interface to generate reflectivity profiles
5. **Modifying constraints**: Disabling conformal roughness for more complex models
6. **Asymmetric hydration**: Modeling supported bilayers with different solvent exposure on each side

The `Bilayer` assembly provides a convenient way to model phospholipid bilayers with physically meaningful constraints, reducing the number of free parameters while maintaining flexibility for complex systems.